# Fine-tune N-ATLAS for high-impact Nigerian sectors

General-purpose models become more useful when adapted to the terminology, data and context of a particular field.

### Target Domains
- Education and learning
- Primary healthcare and community health
- Smallholder agriculture
- Legal rights and civic literacy
- Basic financial literacy

### Requirements
- Fine-tune N-ATLAS for one clearly defined domain
- Demonstrate measurable improvement over base N-ATLAS
- Deploy through a documented API another developer can integrate

**Fine-tuning starter kit with training scripts and evaluation tools**

You can use your local GPU assuming it has 24GB of RAM, or a cloud GPU provider. We will use QLoRA, a fine-tuning method that combines quantization and LoRA. Since it's a large model we will load it in 4bit using `bitsandbytes` and use QLoRA to train using the PEFT library from Hugging Face.

### How to Use This Notebook for Your Sector
To adapt N-ATLaS to a specialized domain, follow these steps:

**1. Format the Domain Data:**
Gather domain-specific documents or Q&A pairs and save them as a single JSONL file (e.g., `my_dataset.jsonl`). Each line should be a JSON object with a `"text"` key containing the training example:
```json
{"text": "User: How do I treat cassava root rot?\nAssistant: Ensure proper soil drainage and apply recommended fungicides during the early stages of planting."}
{"text": "User: What is the penalty for late tax filing in Lagos?\nAssistant: Under the LIRS regulations, late filing incurs a penalty of..."}
```

**2. Update the Dataset Path:**
In the dataset loading cell below, change `"my_dataset.jsonl"` to the path of your specialized dataset.

**3. Execute the Pipeline:**
Run the notebook cells sequentially on an environment with a 24GB GPU. The notebook automatically handles 4-bit quantization, formatting, and LoRA training.

**4. Test the Domain Adaptation:**
At the conclusion of the notebook, modify the `eval_prompt` to a question specific to your dataset (e.g., *"How do I treat cassava root rot?"*) to verify that the model has successfully acquired the specialized terminology and context.

### 1. Preparing data

The training pipeline requires a prepared dataset (e.g., `my_dataset.jsonl`). Adapting the following steps for alternative formats such as JSON or CSV is straightforward using the Hugging Face `datasets` library.

### Load Dataset
Ensure that the execution environment meets the hardware requirements (e.g., a GPU with 24GB of VRAM) to prevent out-of-memory exceptions during training.

In [ ]:
# You only need to run this once per machine
!pip install -q -U bitsandbytes
!pip install -q -U huggingface_hub
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets scipy ipywidgets matplotlib

In [ ]:
from datasets import load_dataset, Features, Value

# Truncating large documents to fit within the model context length using sample_by='paragraph'.
features = Features({ "text": Value("string") })
dataset = load_dataset("text", data_files={"train": "my_dataset.jsonl"}, sample_by="paragraph")['train']

### 2. Load Base Model

Load the base model. Note that access to gated models requires prior approval via a Hugging Face account. 4-bit quantization will be applied to reduce memory consumption while preserving performance.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

base_model_id = "NCAIR1/N-ATLaS"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

login() # Authenticate with Hugging Face to access gated models

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto")

### 3. Tokenization

Initialize the tokenizer. Left-padding is applied to optimize memory usage during training.

Tokenizing without truncation or padding initially allows for analysis of the dataset's length distribution.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token

Select an appropriate `max_length` to truncate and pad training examples. In this configuration, `max_length` is set to 256 tokens.

Re-tokenize the dataset with padding and truncation applied. The tokenization function aligns `labels` with `input_ids`, consistent with standard self-supervised causal language modeling.

In [ ]:
max_length = 256

def generate_and_tokenize_prompt(prompt):
    result = tokenizer(
        prompt['text'],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
tokenized_dataset = dataset.map(generate_and_tokenize_prompt)

Evaluate the model's base performance prior to fine-tuning.

### 4. Set Up LoRA

Apply preprocessing to prepare the quantized model for training using the `prepare_model_for_kbit_training` utility from the PEFT library.

In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

Examine the model architecture. QLoRA will be applied to all linear layers: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`, and `lm_head`.

Define the LoRA configuration.

`r` designates the rank of the low-rank matrix used in the adapters, controlling the number of trained parameters. A higher rank allows for greater expressivity at the cost of computational overhead.

`alpha` serves as the scaling factor for the learned weights. The weight matrix is scaled by `alpha/r`, assigning more weight to the LoRA activations when `alpha` is increased.

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)

### 5. Run Training!

In [ ]:
import transformers
from datetime import datetime

project = "natlas-finetune"
base_model_name = "NCAIR1/N-ATLaS"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,
        max_steps=500,
        learning_rate=2.5e-5, # Lower learning rate optimized for fine-tuning
        bf16=True,
        optim="paged_adamw_8bit",
        save_strategy="steps",       # Save the model checkpoint every logging step
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

### 6. Evaluate the Fine-Tuned Model

The PEFT library saves only the QLoRA adapters. To perform inference, first load the base model from the Hugging Face Hub:

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

login()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, 
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

eval_tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Load the QLoRA adapter from the checkpoint directory associated with the optimal validation performance:

In [ ]:
from peft import PeftModel

ft_model = PeftModel.from_pretrained(base_model, "natlas-finetune/checkpoint-500")

Execute inference:

Run the evaluation prompt to verify the domain adaptation of the fine-tuned model.

In [ ]:
eval_prompt = """ 
User: What are the most effective ways for community health workers to prevent the spread of malaria in rural Nigerian villages?
Assistant: """
model_input = eval_tokenizer(eval_prompt, return_tensors="pt").to("cuda")

ft_model.eval()
with torch.no_grad():
    print(eval_tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=100, repetition_penalty=1.15)[0], skip_special_tokens=True))

### Model Output Verified